# Control policy optimization Mujoco

## Set-up

### Imports

In [1]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

Sat Jun 13 14:41:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.51                 Driver Version: 555.97         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   69C    P0             18W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# @title Import packages for plotting and creating graphics
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [3]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
import jax.numpy as jnp
import jax.random as jr
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

print("These device(s) are detected:", jax.devices())

These device(s) are detected: [CudaDevice(id=0)]


E0613 14:41:26.946042    1852 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0613 14:41:26.968216    1702 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


### Functions

In [4]:
from mujoco_playground import registry

class MujocoGymnaxWrapper:

    def __init__(self, env_name = None, env_instance = None, config_overrides = None):
        if env_instance is not None:
            self.env = env_instance
        elif config_overrides is not None:
            self.env = registry.load(env_name, config_overrides=config_overrides)
        else:
            self.env = registry.load(env_name)
        # self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, key, state, action, params=None):

      action = action.reshape(self.action_space)
      next_state = self.env.step(state, action)

      done = next_state.done
      obs = next_state.obs
      reward = next_state.reward

      return obs, next_state, reward, done, {}

    def render(self, rollout):
      return self.env.render(rollout)

    @property
    def dt(self):
        return self.env.dt

In [5]:
def compute_trajectory_py(key, env, policy, T=1000, stride=1):
  jit_reset = jax.jit(env.reset)
  jit_step = jax.jit(env.step)
  obs, env_state = jit_reset(key)
  states = []

  for t in range(T):
      action = policy(obs)
      obs, env_state, reward, done, _ = jit_step(key, env_state, action)

      if t % stride == 0:
          states.append(env_state)

      if done:
          break

  return states

## Cartpole

In [6]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, num_outputs_per_tree=1, device_type = 'gpu', max_nodes = 31)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 2.67 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 11.04 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 4.31 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 3.92 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 3.23 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8f5355c load on device 'cuda:0' took 11.38 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 2.34 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 72.11 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.solver b116efb load o

In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 117.93 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -365.8791198730469, equation: [-0.0638]
Complexity: 3, fitness: -546.54248046875, equation: [-0.0775*y3]
Complexity: 5, fitness: -979.6781616210938, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -987.0819702148438, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.4639892578125, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -473.13104248046875, equation: [y2]
Complexity: 3, fitness: -697.3924560546875, equation: [0]
Complexity: 5, fitness: -979.6781616210938, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -987.0819702148438, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.4639892578125, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -473.13104248046875, equation: [y2]
Complexity: 3, fitness: -697.3924560546875, equation: [0]
Comple

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 4.4*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, num_outputs_per_tree=1, device_type = 'gpu', max_nodes = 31)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 3.27 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 11.03 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 5.70 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 3.41 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 2.72 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8f5355c load on device 'cuda:0' took 3.51 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 2.03 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 68.27 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.solver b116efb load on

In [ ]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 127.55 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -365.8791198730469, equation: [-0.0638]
Complexity: 3, fitness: -546.54248046875, equation: [-0.0775*y3]
Complexity: 5, fitness: -979.6781616210938, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -987.0819702148438, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.4639892578125, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -473.13104248046875, equation: [y2]
Complexity: 3, fitness: -697.3924560546875, equation: [0]
Complexity: 5, fitness: -979.6781616210938, equation: [y2*(y1 + 0.361)]
Complexity: 6, fitness: -987.0819702148438, equation: [2*y2 + sin(y2)]
Complexity: 7, fitness: -992.4639892578125, equation: [y1**2*y2 + y2]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -473.13104248046875, equation: [y2]
Complexity: 3, fitness: -697.3924560546875, equation: [0]
Comple

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.75*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Reacher

In [6]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy', config_overrides={'njmax': 500, 'naconmax': 500})

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


### 25 generations

In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2, max_nodes = 31)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 2.78 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 9.93 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 4.27 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 2.48 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 1.91 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8f5355c load on device 'cuda:0' took 2.27 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 1.62 ms  (cached)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 73.40 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.solver b116efb load on 

In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 138.36 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -20.25, equation: [-0.515, 0]
Complexity: 2, fitness: -62.5, equation: [sin(y4), 0]
Complexity: 3, fitness: -98.0, equation: [y0*y3, 0]
Complexity: 4, fitness: -189.4375, equation: [y2 + 2*sin(y3), 0]
Complexity: 5, fitness: -383.5625, equation: [2*y2 + y3, 2*y2]
Complexity: 6, fitness: -401.3125, equation: [y2 + sin(y3), 2*y2 + sin(y3)]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -176.1875, equation: [y2, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -189.4375, equation: [y2 + 2*sin(y3), 0]
Complexity: 5, fitness: -383.5625, equation: [2*y2 + y3, 2*y2]
Complexity: 6, fitness: -401.3125, equation: [y2 + sin(y3), 2*y2 + sin(y3)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -176.1875, equation: [y2, 0]
Complexity: 3, fitness: -182.125, equation: [2.08*y2, 0]
Complexity: 

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([2*obs[3]**2 + 5*obs[3], -2.26*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3]*obs[5] - 2.26*obs[3]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [9]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5'].


In [10]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 53.43 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -20.25, equation: [-0.515, 0]
Complexity: 2, fitness: -62.5, equation: [sin(y4), 0]
Complexity: 3, fitness: -98.0, equation: [y0*y3, 0]
Complexity: 4, fitness: -189.4375, equation: [y2 + 2*sin(y3), 0]
Complexity: 5, fitness: -383.5625, equation: [2*y2 + y3, 2*y2]
Complexity: 6, fitness: -401.3125, equation: [y2 + sin(y3), 2*y2 + sin(y3)]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -176.1875, equation: [y2, 0]
Complexity: 3, fitness: -181.875, equation: [2*y2, 0]
Complexity: 4, fitness: -189.4375, equation: [y2 + 2*sin(y3), 0]
Complexity: 5, fitness: -383.5625, equation: [2*y2 + y3, 2*y2]
Complexity: 6, fitness: -401.3125, equation: [y2 + sin(y3), 2*y2 + sin(y3)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -176.1875, equation: [y2, 0]
Complexity: 3, fitness: -182.125, equation: [2.08*y2, 0]
Complexity: 4

### Simulation

In [ ]:
stride = 2

mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([obs[3]*obs[5] + 6*obs[3], -4.53*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3] - 0.245])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Swimmer

In [11]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)

In [12]:
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)

### 25 generations

In [16]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12'].


In [17]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 72.24 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -153.64163208007812, equation: [0.639, 0]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 7, fitness: -168.04928588867188, equation: [y3*(y0 + y8) + y7, 0]
Complexity: 9, fitness: -173.86871337890625, equation: [y2 + (y1 + y12)*(y12 + y8), 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -163.96102905273438, equation: [y3, 0]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 6, fitness: -166.5493621826172, equation: [sin(y10*y6 + y2), 0]
Complexity: 7, fitness: -174.7647705078125, equation: [y12*y9 + sin(cos(y2)), cos(y2)]
Complexity: 8, fitness: -181.16119384765625, equation: [y2 + (y1 + y12)*cos(y0), cos(y0)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 4, fitness: -166.54429626464844, equation:

### Simulation

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim
stride = 2

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-obs[11] - obs[2] + jnp.sin(jnp.sin(2*obs[9])), obs[1]*obs[3] + obs[2] + (2*obs[0] + obs[2])*(-obs[3] + obs[4])])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [18]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)

In [19]:
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)

In [20]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=2, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12'].


In [21]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 68.92 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -153.64163208007812, equation: [0.639, 0]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 7, fitness: -168.04928588867188, equation: [y3*(y0 + y8) + y7, 0]
Complexity: 9, fitness: -173.86871337890625, equation: [y2 + (y1 + y12)*(y12 + y8), 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -163.96102905273438, equation: [y3, 0]
Complexity: 2, fitness: -166.4177703857422, equation: [sin(y2), 0]
Complexity: 6, fitness: -166.5493621826172, equation: [sin(y10*y6 + y2), 0]
Complexity: 7, fitness: -174.7647705078125, equation: [y12*y9 + sin(cos(y2)), cos(y2)]
Complexity: 8, fitness: -181.16119384765625, equation: [y2 + (y1 + y12)*cos(y0), cos(y0)]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -166.49172973632812, equation: [y2, 0]
Complexity: 4, fitness: -166.54429626464844, equation:

### Simulation

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim
stride = 2
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7] + obs[9] + jnp.sin(obs[2] - obs[6]), obs[0]*obs[2] + 2*obs[0] + 2*obs[1]*obs[3]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Hopper

In [6]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


### 25 generations

In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=4, max_nodes = 31)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 6.45 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 65.54 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 22.44 ms  (cached)
Module _nxn_broadphase__locals__kernel_c763f4cb c763f4c load on device 'cuda:0' took 16.10 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_052aeb65 052aeb6 load on device 'cuda:0' took 18.34 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 37.55 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 40.64 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 36.89 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 18a78e5 load on dev

In [8]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 151.00 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -5.3510069847106934e-05, equation: [0.639, 0, 0, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.6095319390296936, equation: [0, y10 + y14, 0, 0]
Complexity: 10, fitness: -0.6918385624885559, equation: [y0*y1, 0.935*y0*y1*(y11 + sin(y12)) + sin(y12), y11 + sin(y12), 0]
Complexity: 11, fitness: -1.1942371129989624, equation: [0, y10 - y11*(y14 + y4) + 2*y14, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -0.08202669024467468, equation: [0, 0, y8, 0]
Complexity: 3, fitness: -0.7063678503036499, equation: [0, y10 + y5, 0, 0]
Complexity: 9, fitness: -0.7179629802703857, equation: [0, y10 - y11*(y14 + y4) + y4, 0, 0]
Complexity: 11, fitness: -1.579916000366211, equation: [0, y10 - y11*(y14 + y4) + 2*y14, 0, 0]
invalid solutions: 0 0
In generation 3
Complexity: 1, fit

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[4], 2*obs[0] + obs[8]*(obs[8] - obs[9]), 2*obs[1] - obs[2] + obs[5], obs[1] + 2*obs[8]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [9]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=4, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14'].


In [10]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 55.13 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -5.3510069847106934e-05, equation: [0.639, 0, 0, 0]
Complexity: 2, fitness: -0.05380568653345108, equation: [0, 0, sin(y3), 0]
Complexity: 3, fitness: -0.5982257127761841, equation: [0, y10 + y14, 0, 0]
Complexity: 9, fitness: -0.7498444318771362, equation: [0, y10 + y5, 0, y10 + y3 + y5 + y6 + y8]
Complexity: 11, fitness: -1.371050238609314, equation: [0, y10 - y11*(y14 + y4) + 2*y14, 0, 0]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -0.08202669024467468, equation: [0, 0, y8, 0]
Complexity: 3, fitness: -0.7462608814239502, equation: [0, y10 + y5, 0, 0]
Complexity: 5, fitness: -0.8373975157737732, equation: [0, y1 - y11 - 0.66, 0, 0]
Complexity: 11, fitness: -1.3813989162445068, equation: [0, y10 + y5, y1*y5, y1*y5 + y10 + y3 + y5 + y6]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitness: -0.08202669024467468, e

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[2]*(-2.43*obs[2] - 1.43), obs[10]*(obs[10] - 2*obs[12] + 2*obs[4]) + obs[10] - obs[6], 2*obs[1] - obs[2] - obs[6]**2 - obs[6] + obs[8] + 0.626, 2*obs[1] + obs[13] + 3*obs[4]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Walker

In [11]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')

In [12]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6, max_nodes = 31)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_cdb981f2 c75d2e6 load on device 'cuda:0' took 122.04 ms  (cached)
Module solve_init_jaref__locals__kernel_051c9471 051c947 load on device 'cuda:0' took 2.73 ms  (cached)
Module mul_m_dense__locals___mul_m_dense_86090c8a 86090c8 load on device 'cuda:0' took 2.15 ms  (cached)
Module update_constraint_gauss_cost__locals__kernel_0aec6e48 0aec6e4 load on device 'cuda:0' took 2.85 ms  (cached)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_2b23959a 84da6db load on device 'cuda:0' took 2.95 ms  (cached)
Module update_gradient_cholesky__locals__kernel_97d1b30b 6b32db2 load on device 'cuda:0' took 122.21 ms  (cached)
Module linesearch_iterative__locals__kernel_6962ca73 89f17c9 load on device 'cuda:0' took 4.58 ms  (cached)
Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16', 'y17', 'y18', 'y19', 'y20', 'y21', 'y22', 'y23'].

In [13]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 92.85 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -21.106983184814453, equation: [0, 0, 0, 0, 0.839, 0]
Complexity: 2, fitness: -23.796567916870117, equation: [0, cos(y3), 0, 0, 0, 0]
Complexity: 3, fitness: -57.8367919921875, equation: [0, 0, 0, y19 + y21, 0, 0]
Complexity: 7, fitness: -64.22988891601562, equation: [0, 0, 0, y18 + y4, 0, 1.39*y15 + 1.39*y18 + 1.39*y4]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -66.87385559082031, equation: [y18, 0, 0, 0, 0, 0]
Complexity: 11, fitness: -68.97151184082031, equation: [0, y11 + (y11 - 0.66)*(y17 - y22 + y5 + y7) - 0.66, -y22 + y7, y17 - y22 + y5 + y7, y17 + y5, 0]
Complexity: 12, fitness: -75.25572967529297, equation: [0, 0, 0, (y23 + y6)*(y22 + y3 + cos(y17**2)) + cos(y17**2), y22 + y3, y23 + y6]
Complexity: 18, fitness: -97.53541564941406, equation: [(y16 + y20)*sin(y11), y20 + y22, 0, y18 + y4 + sin(y11), (y16 + y20)*si

### Simulation

In [ ]:
stride = 3
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([obs[15] + 2*obs[18] + obs[21], obs[9]*(-0.799*obs[19] - 0.799*obs[21]), obs[1] + obs[15], 2*obs[18] + jnp.cos(obs[14]), obs[1]*obs[10]*obs[17]**2, obs[11] + obs[14] + obs[5]])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [16]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16', 'y17', 'y18', 'y19', 'y20', 'y21', 'y22', 'y23'].


In [15]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 109.54 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -21.106773376464844, equation: [0, 0, 0, 0, 0.839, 0]
Complexity: 2, fitness: -23.972570419311523, equation: [0, 0, 0, sin(y18), 0, 0]
Complexity: 3, fitness: -53.069793701171875, equation: [0, 0, 0, y19 + y21, 0, 0]
Complexity: 7, fitness: -63.02339172363281, equation: [0, 0, 0, y18 + y4, 0, 1.39*y15 + 1.39*y18 + 1.39*y4]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -27.943172454833984, equation: [0, 0, 0, y12, 0, 0]
Complexity: 3, fitness: -64.45469665527344, equation: [y18 + y4, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -65.09431457519531, equation: [y16 + y21, 0, 0, 0, y12*(0.215*y16 + 0.215*y21), 0]
Complexity: 8, fitness: -70.35066223144531, equation: [0, sin(y21), 0, y18 + y4, 0, 1.39*y18 + 1.39*y4 + 1.39*sin(y21)]
Complexity: 11, fitness: -70.51624298095703, equation: [y15*y9*(y18 + y23 - 0.452) + y18 + y23 - 0.452, 

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([3*obs[18] - 0.455, obs[10] - obs[13] - obs[19] + obs[22], 2*obs[17], obs[18], obs[0] + obs[14] - 2*obs[8], 2*obs[20]**2])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Cheetah

In [6]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 12.5
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce GTX 1650" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/theun/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [7]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 25
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6, max_nodes = 31)

These device(s) are detected: [CudaDevice(id=0)]
Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 2.84 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 26bbc02 load on device 'cuda:0' took 16.31 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 10.42 ms  (cached)
Module _nxn_broadphase__locals__kernel_c763f4cb c763f4c load on device 'cuda:0' took 3.75 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_052aeb65 052aeb6 load on device 'cuda:0' took 2.93 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint c567367 load on device 'cuda:0' took 4.40 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 16fbac8 load on device 'cuda:0' took 3.39 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 8bfe11c load on device 'cuda:0' took 2.84 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.sensor 18a78e5 load on device '

In [19]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 67.82 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -1.322232961654663, equation: [0, 0, 0, 0, 0.839, 0]
Complexity: 2, fitness: -15.58510971069336, equation: [0, 0, 0, 0, 0, sin(y13)]
Complexity: 3, fitness: -57.53268051147461, equation: [y8 - 0.826, 0, 0, 0, 0, 0]
Complexity: 5, fitness: -85.38514709472656, equation: [2*y11 + 0.185, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -136.58786010742188, equation: [y0 - y1 + y15 + y9, 0, 0, 0, 0, -y1 + y15]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -14.988991737365723, equation: [0, 0, 0, 0, y11, 0]
Complexity: 2, fitness: -15.58510971069336, equation: [0, 0, 0, 0, 0, sin(y13)]
Complexity: 3, fitness: -57.53268051147461, equation: [y8 - 0.826, 0, 0, 0, 0, 0]
Complexity: 4, fitness: -95.99891662597656, equation: [y14 + 2*sin(y9), 0, 0, 0, 0, 0]
Complexity: 5, fitness: -148.84092712402344, equation: [-y13 + 2*y16 + 2*y9, 0, 0, 0, 0, 

### Simulation

In [ ]:
stride = 3
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([(obs[11] + jnp.cos(obs[11]))*jnp.cos(obs[5]), obs[2]**3*obs[5], obs[11]*obs[2] + obs[11] - obs[2], 0, -obs[1] + obs[3]*obs[8] + obs[3], obs[10] + jnp.cos(jnp.cos(jnp.cos(obs[6])))])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

### 100 generations

In [8]:
from kozax.genetic_programming import GeneticProgramming
from kozax.fitness_functions.Gymnax_fitness_function import GymFitnessFunction

#Define hyperparameters
population_size = 100
num_populations = 5
num_generations = 100
batch_size = 16

fitness_function = GymFitnessFunction.__new__(GymFitnessFunction)
fitness_function.env = mujoco_env
#in Gymnax, env_params are max_speed: float = 8.0, max_torque: float = 2.0, dt: float = 0.05, g: float = 10.0, m: float = 1.0, l: float = 1.0, max_steps_in_episode: int = 200
fitness_function.env_params = None
fitness_function.num_steps = 1000

#Define operators and variables
operator_list = [
    ("+", lambda x, y: jnp.add(x, y), 2, 0.5),
    ("-", lambda x, y: jnp.subtract(x, y), 2, 0.1),
    ("*", lambda x, y: jnp.multiply(x, y), 2, 0.5),
    ("sin", lambda x: jnp.sin(x), 1, 0.1),
    ("cos", lambda x: jnp.cos(x), 1, 0.1),
]

obs, _ = fitness_function.env.reset(jax.random.PRNGKey(0))
variable_list = [[f"y{i}" for i in range(obs.shape[0])]]

#Initialize strategy
strategy = GeneticProgramming(num_generations, population_size, fitness_function, operator_list, variable_list, num_populations=num_populations, device_type = 'gpu', num_outputs_per_tree=6, max_nodes = 31)

Input data should be formatted as: ['y0', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16'].


In [9]:
key = jr.PRNGKey(0)
data_key, gp_key = jr.split(key, 2)

# The data comprises keys need to initialize the batch of environments.
batch_keys = jr.split(data_key, batch_size)

strategy.fit(gp_key, batch_keys, verbose=1)

Compiling code for evaluation and evolution...
Finished compilation in 152.20 seconds
invalid solutions: 0 0
In generation 1
Complexity: 1, fitness: -1.3221333026885986, equation: [0, 0, 0, 0, 0.839, 0]
Complexity: 2, fitness: -15.578259468078613, equation: [0, 0, 0, 0, 0, sin(y13)]
Complexity: 3, fitness: -57.74143600463867, equation: [y8 - 0.826, 0, 0, 0, 0, 0]
Complexity: 5, fitness: -89.89989471435547, equation: [2*y11 + 0.185, 0, 0, 0, 0, 0]
Complexity: 7, fitness: -129.48260498046875, equation: [y0 - y1 + y15 + y9, 0, 0, 0, 0, -y1 + y15]
invalid solutions: 0 0
In generation 2
Complexity: 1, fitness: -15.059751510620117, equation: [0, 0, 0, 0, y11, 0]
Complexity: 2, fitness: -15.578259468078613, equation: [0, 0, 0, 0, 0, sin(y13)]
Complexity: 3, fitness: -157.8468475341797, equation: [y16 + y7, 0, 0, 0, 0, 0]
Complexity: 11, fitness: -160.36380004882812, equation: [y0*y10 + y0 - y1 + y15 + y9, 0, y0*y10, 0, 0, -y1 + y15]
invalid solutions: 0 0
In generation 3
Complexity: 1, fitnes

### Simulation

In [ ]:
stride = 2
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([-obs[0] - 2*obs[12] + obs[16], jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[4])))))), obs[10] + 3*obs[11], 0.334*obs[4] + 0.112, obs[12], jnp.cos(jnp.cos(3*obs[7]))])
key = jr.PRNGKey(0)
state_history = compute_trajectory_py(key, mujoco_env, policy, stride = stride)

In [ ]:
frames = mujoco_env.render(state_history)
media.show_video(frames, fps=1.0 / mujoco_env.dt / stride)

## Multiple evaluations

In [ ]:
T = 1000

def repeated_evaluation(key, policy, env):
    def single_run(key):
        obs, env_state = env.reset(key)
        (_, _, _, _), (_, treward, _, _) = jax.lax.scan(
            step_fn(policy, env),
            (key, obs, env_state, False),
            jnp.arange(T)
        )
        return jnp.sum(treward)

    keys = jr.split(key, 10)
    return jax.vmap(single_run)(keys)

def step_fn(policy, env):

    def _step(carry, t):
        key, obs, env_state, done = carry
        action = policy(obs)
        key, subkey = jr.split(key)
        obs, env_state, reward, done_new, _ = env.step(
            subkey,
            env_state,
            action,
            None
        )

        return (key, obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

### CartPole

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 4.4*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.75*obs[2] + obs[3] + 2*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Reacher

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([2*obs[3]**2 + 5*obs[3], -2.26*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3]*obs[5] - 2.26*obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([obs[3]*obs[5] + 6*obs[3], -4.53*obs[0]*obs[3] - 4.53*obs[2] - 2.26*obs[3] - 0.245])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Swimmer

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-obs[11] - obs[2] + jnp.sin(jnp.sin(2*obs[9])), obs[1]*obs[3] + obs[2] + (2*obs[0] + obs[2])*(-obs[3] + obs[4])])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7] + obs[9] + jnp.sin(obs[2] - obs[6]), obs[0]*obs[2] + 2*obs[0] + 2*obs[1]*obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Hopper

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[4], 2*obs[0] + obs[8]*(obs[8] - obs[9]), 2*obs[1] - obs[2] + obs[5], obs[1] + 2*obs[8]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([obs[2]*(-2.43*obs[2] - 1.43), obs[10]*(obs[10] - 2*obs[12] + 2*obs[4]) + obs[10] - obs[6], 2*obs[1] - obs[2] - obs[6]**2 - obs[6] + obs[8] + 0.626, 2*obs[1] + obs[13] + 3*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Walker

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([obs[15] + 2*obs[18] + obs[21], obs[9]*(-0.799*obs[19] - 0.799*obs[21]), obs[1] + obs[15], 2*obs[18] + jnp.cos(obs[14]), obs[1]*obs[10]*obs[17]**2, obs[11] + obs[14] + obs[5]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([3*obs[18] - 0.455, obs[10] - obs[13] - obs[19] + obs[22], 2*obs[17], obs[18], obs[0] + obs[14] - 2*obs[8], 2*obs[20]**2])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

### Cheetah

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([(obs[11] + jnp.cos(obs[11]))*jnp.cos(obs[5]), obs[2]**3*obs[5], obs[11]*obs[2] + obs[11] - obs[2], 0, -obs[1] + obs[3]*obs[8] + obs[3], obs[10] + jnp.cos(jnp.cos(jnp.cos(obs[6])))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([-obs[0] - 2*obs[12] + obs[16], jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[4])))))), obs[10] + 3*obs[11], 0.334*obs[4] + 0.112, obs[12], jnp.cos(jnp.cos(3*obs[7]))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)